# Data Prep Handlers

### clean_damage_data.py
Script to take raw damage shapefile and preprocess it to get to the format needed by our architecture (see below).

Damage

    Name: [city]_damage.gpkg
    Variables (integer):
    Names: [yyyy-mm-dd] (one per census)
    Values:
        1 low destruction
        2 medium destruction
        3 high destruction
        0 otherwise

To run, navigate to project root and run the following: 
```
python dev/clean_damage_data.py ../data/homs/others/homs_damage.gpkg --suffix _processed
```

In [33]:
%%writefile clean_damage_data.py

import geopandas as gpd
import numpy as np
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("input_file_path", help="Path to the input raw damage .gpkg")
parser.add_argument("--suffix", help="Suffix to add to output filename (default: '')")


args = parser.parse_args()
INPUT_PATH = args.input_file_path
SUFFIX = ""


if args.suffix:
    SUFFIX = args.suffix



df = gpd.read_file(INPUT_PATH)
sensor_date_columns = [col for col in df.columns if 'SensDt' in col]
damage_class_columns = [col for col in df.columns if 'DmgCls' in col and 'GrpDmgCls' not in col ]


for i, sensor_date_col in enumerate(sensor_date_columns):
    if i==0:
        allDates = df[sensor_date_col]
    else:
        allDates = allDates.append(df[sensor_date_col])
        
sensor_date_values = allDates.unique()
sensor_date_values = sensor_date_values[sensor_date_values != np.array(None)]

new_df = []
for i, row in df.iterrows():

    row_entry = {}
    row_entry['geometry'] = row['geometry']
    
    for j, sensor_date_col in enumerate(sensor_date_columns):
        if(row[sensor_date_col] != None):
            row_entry[row[sensor_date_col]] = row[damage_class_columns[j]]
    
    new_df.append(row_entry)


df = gpd.GeoDataFrame(new_df)
df = df.replace({np.nan: 0, "Destroyed": 3, "Severe Damage": 2, "Moderate Damage": 1})
print("Unique_values: {}".format(df[df.columns[1]].unique()))
df.to_file(INPUT_PATH.split(".gpkg")[0] + SUFFIX + ".gpkg", driver="GPKG")

Overwriting clean_damage_data.py


### clean_noanalysis_data.py
Script to take raw damage shapefile and preprocess it to get to the format needed by our architecture (see below).

Noanalysis

    Name: [city]_noanalysis.gpkg
    Variables (string): reason

To run, navigate to project root and run the following: 
```
python dev/clean_noanalysis_data.py ../data/homs/others/homs_noanalysis.gpkg --suffix _processed
```

In [34]:
%%writefile clean_noanalysis_data.py
import geopandas as gpd
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("input_file_path", help="Path to the input raw damage .gpkg")
parser.add_argument("--suffix", help="Suffix to add to output filename (default: '')")


args = parser.parse_args()
INPUT_PATH = args.input_file_path
SUFFIX = ""

if args.suffix:
    SUFFIX = args.suffix


df = gpd.read_file(INPUT_PATH)
df.columns = ['reason', 'geometry']
df.to_file(INPUT_PATH.split(".gpkg")[0] + SUFFIX + ".gpkg", driver="GPKG")


Overwriting clean_noanalysis_data.py


### clean_settlement_data.py
Script to take raw damage shapefile and preprocess it to get to the format needed by our architecture (see below).

Settlement

    Name: [city]_settlement.gpkg
    Variables: None

To run, navigate to project root and run the following: 
```
python dev/clean_settlement_data.py ../data/homs/others/homs_settlement.gpkg --suffix _processed

In [35]:
%%writefile clean_settlement_data.py
import geopandas as gpd
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--suffix", help="Suffix to add to output filename (default: '')")


args = parser.parse_args()
INPUT_PATH = args.input_file_path
SUFFIX = ""

if args.suffix:
    SUFFIX = args.suffix


# INPUT_PATH = "/Users/arogyak/projects/mwd/data/daraa/others/daraa_settlement.gpkg"
df = gpd.read_file(INPUT_PATH)
df = df.filter(['geometry'])
df.to_file(INPUT_PATH.split(".gpkg")[0] + SUFFIX + ".gpkg", driver="GPKG")


Overwriting clean_settlement_data.py


## Working with preprocessing.py

In [ ]:
# Modules
import numpy as np
import geopandas
import rasterio
import re

from numpy import random
from os import path
from rasterio import features
from destruction_utilities import *

In [ ]:
def tiled_profile(source:str, tile_size:tuple=(128, 128, 1)) -> dict:
    '''Computes raster profile for tiles'''
    raster  = rasterio.open(source)
    profile = raster.profile
    assert profile['width']  % tile_size[0] == 0, 'Invalid dimensions'
    assert profile['height'] % tile_size[1] == 0, 'Invalid dimensions'
    affine  = profile['transform']
    affine  = rasterio.Affine(affine[0] * tile_size[0], affine[1], affine[2], affine[3], affine[4] * tile_size[1], affine[5])
    profile.update(width=profile['width'] // tile_size[0], height=profile['height'] // tile_size[0], count=tile_size[2], transform=affine)
    return profile


In [ ]:
# Files
image      = search_data(pattern(city='aleppo', type='image'))[0]
settlement = search_data('aleppo_settlement.*gpkg$')
noanalysis = search_data('aleppo_noanalysis.*gpkg$')

In [ ]:
# Computes analysis zone
profile    = tiled_profile(image, tile_size=(128, 128, 1))
settlement = rasterise(settlement, profile, dtype='bool')
noanalysis = rasterise(noanalysis, profile, dtype='bool')
analysis   = np.logical_and(settlement, np.invert(noanalysis))
del image, settlement, noanalysis

In [ ]:
random.seed(1)
index   = dict(training=0.70, validation=0.15, test=0.15)
index   = np.random.choice(np.arange(len(index)) + 1, np.sum(analysis), p=list(index.values()))
samples = analysis.astype(int)
np.place(samples, analysis, index)
write_raster(samples, profile, '../data/aleppo/vectors/aleppo_samples.tif', nodata=-1, dtype='int8')
del index, samples, analysis

In [ ]:
damage = search_data('aleppo_damage.*gpkg$')
damage = geopandas.read_file(damage)

In [ ]:
damage.plot()

In [ ]:
dates = search_data(pattern(city='aleppo', type='image'))
dates = extract(dates, '\d{4}-\d{2}-\d{2}')

In [ ]:
damage

In [ ]:
dates

In [ ]:
damage[list(set(dates) - set(damage.columns))] = np.nan
damage = damage.reindex(sorted(damage.columns), axis=1)
values = damage.drop(columns='geometry').T
values.fillna(method='ffill', inplace=True)
values.fillna(method='bfill', inplace=True)
damage = geopandas.GeoDataFrame(values.T.astype(int), geometry=damage.geometry)
damage = damage[dates + ['geometry']] # Drops dates not in images
del values


In [ ]:
for date in dates:
    print(date)
    subset = damage[[date, 'geometry']].sort_values(by=date) # Sorting takes the max per pixel
    subset = rasterise(subset, profile, date)
    print(date , subset.shape)
#     write_raster(subset, profile, f'../data/aleppo/labels/label_{date}.tif', nodata=-1, dtype='int8')
# del dates, date, subset

In [ ]:
damage[['2011-01-01', 'geometry']]

In [ ]:
damage